In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("https://raw.githubusercontent.com/Laxminarayen/InceptezGenAI-Batch26/refs/heads/main/27.%20Class%20Imbalance-F1Score-Streamlit/predictive_maintenance.csv")
print(df.shape)

(10000, 10)


In [2]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


In [3]:
df['Failure Type'].value_counts()

,count
Failure Type,
No Failure,9652
Heat Dissipation Failure,112
Power Failure,95
Overstrain Failure,78
Tool Wear Failure,45
Random Failures,18


In [4]:
df.columns

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Target', 'Failure Type'],
      dtype='object')

In [5]:
df = df.drop(columns = ['UDI','Product ID'])

In [6]:
df.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Target',
       'Failure Type'],
      dtype='object')

In [7]:
df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,M,298.1,308.6,1551,42.8,0,0,No Failure
1,L,298.2,308.7,1408,46.3,3,0,No Failure
2,L,298.1,308.5,1498,49.4,5,0,No Failure
3,L,298.2,308.6,1433,39.5,7,0,No Failure
4,L,298.2,308.7,1408,40.0,9,0,No Failure


In [8]:
df['Target'].value_counts() #0-No failure | 1-failure

,count
Target,
0,9661
1,339


In [9]:
df['Failure Type'].value_counts() #No Failure = 0 (Optional preprocessing step drop mismatch rows..)

,count
Failure Type,
No Failure,9652
Heat Dissipation Failure,112
Power Failure,95
Overstrain Failure,78
Tool Wear Failure,45
Random Failures,18


In [10]:
df.loc[(df['Failure Type'] == 'No Failure') & (df['Target'] == 1)] #I ll drop only if part of major class

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
1437,H,298.8,309.9,1439,45.2,40,1,No Failure
2749,M,299.7,309.2,1685,28.9,179,1,No Failure
4044,M,301.9,310.9,1419,47.7,20,1,No Failure
4684,M,303.6,311.8,1421,44.8,101,1,No Failure
5536,M,302.3,311.8,1363,54.0,119,1,No Failure
5941,L,300.6,310.7,1438,48.5,78,1,No Failure
6478,L,300.5,309.8,1663,29.1,145,1,No Failure
8506,L,298.4,309.6,1710,27.3,163,1,No Failure
9015,L,297.2,308.1,1431,49.7,210,1,No Failure


In [11]:
mismatch = (((df['Target']==0) & (df['Failure Type'] != "No Failure")) | ((df['Target']==1) & (df['Failure Type'] == "No Failure")))

In [12]:
mismatch

,0
0,False
1,False
2,False
3,False
4,False
...,...
9995,False
9996,False
9997,False
9998,False


In [13]:
df = df[~mismatch].reset_index(drop=True)

In [14]:
df.shape

(9973, 8)

In [15]:
df['Failure Type'].value_counts()

,count
Failure Type,
No Failure,9643
Heat Dissipation Failure,112
Power Failure,95
Overstrain Failure,78
Tool Wear Failure,45


In [16]:
df = pd.get_dummies(df, columns = ['Type'], drop_first = True)
df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,No Failure,False,True
1,298.2,308.7,1408,46.3,3,0,No Failure,True,False
2,298.1,308.5,1498,49.4,5,0,No Failure,True,False
3,298.2,308.6,1433,39.5,7,0,No Failure,True,False
4,298.2,308.7,1408,40.0,9,0,No Failure,True,False


In [16]:
# [yes, may, no]
# [1,   0,    0] - Category
# [1,   0]        -

# [0,   0,   1] - Rejected
# [0,   0]

# "Degrees of Freedom" #n-1 | 1

In [18]:
df.columns

Index(['Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Target',
       'Failure Type', 'Type_L', 'Type_M'],
      dtype='object')

In [20]:
from sklearn.model_selection import train_test_split

feature_cols = ['Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_L', 'Type_M']

X = df[feature_cols]
y_binary = df['Target']
y_multi = df['Failure Type']

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X,y_multi, test_size = 0.2, random_state = 24, stratify = y_multi)

In [57]:
y_train.value_counts()

,count
Failure Type,
No Failure,7714
Heat Dissipation Failure,90
Power Failure,76
Overstrain Failure,62
Tool Wear Failure,36


In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),columns = feature_cols, index = X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),columns = feature_cols, index = X_test.index)

In [37]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report)
#results = []

def evaluate(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred,'macro')
    f1 = f1_score(y_true, y_pred,'weighted')
    return [accuracy, precision, recall, f1]

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [26]:
results_multi_class = []

log_reg_m = LogisticRegression(random_state=24,max_iter=1000)
Knn_m = KNeighborsClassifier(n_neighbors = 5)
tree_m = DecisionTreeClassifier(random_state=24)

In [27]:
labels = sorted(y_multi.unique())

In [28]:
labels

['Heat Dissipation Failure',
 'No Failure',
 'Overstrain Failure',
 'Power Failure',
 'Tool Wear Failure']

In [45]:
for name, model in [('Logistic Regression', log_reg_m), ('KNN', Knn_m), ('Decision Tree', tree_m)]:
    model.fit(X_train_scaled, y_train)
    print(f"Model: {name}")
    y_pred = model.predict(X_test_scaled)
    #row = evaluate(y_test,y_pred)
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test,y_pred,average='macro'),
        "Recall": recall_score(y_test,y_pred, average='macro'),
        "F1 Score": f1_score(y_test,y_pred, average='weighted')
    }
    print(classification_report(y_test,y_pred,target_names=labels))
    results_multi_class.append(metrics)

Model: Logistic Regression
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.88      0.32      0.47        22
              No Failure       0.98      1.00      0.99      1929
      Overstrain Failure       0.91      0.62      0.74        16
           Power Failure       0.83      0.79      0.81        19
       Tool Wear Failure       0.00      0.00      0.00         9

                accuracy                           0.98      1995
               macro avg       0.72      0.55      0.60      1995
            weighted avg       0.98      0.98      0.98      1995

Model: KNN
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.40      0.09      0.15        22
              No Failure       0.97      1.00      0.98      1929
      Overstrain Failure       0.57      0.25      0.35        16
           Power Failure       0.60      0.16      0.25        19
       Tool Wear Failure       0.0

In [45]:
#Across all failure categories a balance of good predictions is achieved in Decision Tree model as compared a
#against other models.. and the support for this is given by the F1-score (harmonic mean) for individual classes

In [46]:
smallest_class_size = y_train.value_counts().min()
smallest_class_size

36

In [50]:
y_train.value_counts()

,count
Failure Type,
No Failure,7714
Heat Dissipation Failure,90
Power Failure,76
Overstrain Failure,62
Tool Wear Failure,36


In [48]:
safe_k = max(1, min(5, int(smallest_class_size-1)))

In [49]:
safe_k

5

In [51]:
!pip install imblearn

In [53]:
import imblearn

'minority': resample only the minority class;

'not minority': resample all classes but the minority class;

'not majority': resample all classes but the majority class;

'all': resample all classes;

'auto': equivalent to 'not majority'.

In [54]:
from  imblearn.over_sampling  import SMOTE

smote = SMOTE(random_state=24, k_neighbors = safe_k)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

In [55]:
y_train_sm.value_counts()

#No failure has to be passed so other categories no what is majority number

,count
Failure Type,
No Failure,7714
Heat Dissipation Failure,7714
Overstrain Failure,7714
Power Failure,7714
Tool Wear Failure,7714


In [56]:
for name, model in [('Logistic Regression', log_reg_m), ('KNN', Knn_m), ('Decision Tree', tree_m)]:
    model.fit(X_train_sm, y_train_sm) #Fitting on smote
    print(f"Model: {name}")
    y_pred = model.predict(X_test_scaled) #Test - real life simulation
    #row = evaluate(y_test,y_pred)
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test,y_pred,average='macro'),
        "Recall": recall_score(y_test,y_pred, average='macro'),
        "F1 Score": f1_score(y_test,y_pred, average='weighted')
    }
    print(classification_report(y_test,y_pred,target_names=labels))
    results_multi_class.append(metrics)

Model: Logistic Regression
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.29      0.91      0.43        22
              No Failure       1.00      0.88      0.93      1929
      Overstrain Failure       0.42      0.94      0.58        16
           Power Failure       0.38      0.95      0.55        19
       Tool Wear Failure       0.04      0.67      0.08         9

                accuracy                           0.88      1995
               macro avg       0.42      0.87      0.51      1995
            weighted avg       0.97      0.88      0.92      1995

Model: KNN
                          precision    recall  f1-score   support

Heat Dissipation Failure       0.29      0.68      0.41        22
              No Failure       0.99      0.93      0.96      1929
      Overstrain Failure       0.42      0.88      0.57        16
           Power Failure       0.34      0.68      0.46        19
       Tool Wear Failure       0.0

In [ ]:
#Hyper parameter tuning
#Feature Engineering